# Capital Allocation Analyst — Google ADK + Gemini 2.5 (Vertex AI) + BigQuery

A re-implementation of `financial_analyst_from_scratch.ipynb` on top of the
**Google Agent Development Kit (ADK)**, tuned for a **capital allocation**
process: comparing competing projects / segments / assets on a consistent
basis (NPV, IRR, payback, ROIC/RAROC, return spread vs. cost of capital) and
recommending how to deploy a limited capital budget.

It reads the inputs straight from a **BigQuery** warehouse and does the
investment math with the ported `calc` tool.

What changes vs. the original notebook:

| Concern | `financial_analyst_from_scratch.ipynb` | this notebook |
|---|---|---|
| Agent loop / tool-calling | hand-coded `agent_loop` over Ollama | provided by ADK (`LlmAgent` + `Runner`) |
| Model | local `qwen3:32b` via Ollama | **Gemini 2.5 Pro via Vertex AI** (thinking on) |
| Data | Tavily web + Finnhub | **BigQuery only** (ADK first-party toolset) |
| Planning / memory | custom todo + task graph + compaction | ADK sessions + Gemini thinking |
| Math | `calc` (sandboxed) | **`calc` preserved** for NPV/IRR/payback/etc. |

**Tools:** the BigQuery toolset (data) + `calc` (investment math). No web/market
tools — capital-allocation inputs come from the warehouse.

**Safety posture:** BigQuery access is **read-only** (`WriteMode.BLOCKED`). Point
it at a financial institution's warehouse only with least-privilege credentials.
Every query bills against your own Vertex/BQ project.

---

### Setup

```bash
pip install google-adk google-cloud-bigquery

# ADC drives BOTH Vertex AI and BigQuery:
gcloud auth application-default login
#   ...or: export GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json

export GOOGLE_CLOUD_PROJECT=your-gcp-project
export GOOGLE_CLOUD_LOCATION=us-central1
export GOOGLE_GENAI_USE_VERTEXAI=TRUE   # the config cell sets this if unset
```


In [ ]:
# Optional: install dependencies (uncomment on first run)
# %pip install -q google-adk google-cloud-bigquery

In [ ]:
"""
Imports + configuration. All knobs live here.
"""
from __future__ import annotations

import asyncio
import logging
import math
import os
import uuid

# Route google-genai through Vertex AI rather than the public Gemini API.
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "TRUE")

PROJECT  = os.environ.get("GOOGLE_CLOUD_PROJECT", "")          # set this
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", LOCATION)

MODEL    = os.environ.get("FIN_AGENT_MODEL", "gemini-2.5-pro")
APP_NAME = "financial_analyst_adk"
USER_ID  = os.environ.get("FIN_AGENT_USER", "analyst")

# Cap how many rows execute_sql returns into context (cost / context hygiene).
BQ_MAX_RESULT_ROWS = int(os.environ.get("BQ_MAX_RESULT_ROWS", "200"))

# Gemini 2.5 "thinking" budget in tokens. -1 = let the model decide dynamically.
THINKING_BUDGET = int(os.environ.get("FIN_THINKING_BUDGET", "-1"))

logging.basicConfig(
    level=os.environ.get("AGENT_LOG_LEVEL", "INFO").upper(),
    format="[%(levelname)s] %(name)s | %(message)s",
)
log = logging.getLogger("fin_adk")
log.info("model=%s location=%s vertexai=%s", MODEL, LOCATION,
         os.environ.get("GOOGLE_GENAI_USE_VERTEXAI"))

## `calc` — sandboxed financial-math tool

Ported verbatim from the original notebook. The model passes a Python expression;
we `eval` it with `__builtins__` stripped and a curated namespace (every `math.*`
function plus Excel-style finance helpers). Exposed to ADK as a plain function —
ADK reads the type hints + docstring to build the tool schema automatically.

In [ ]:
def _npv(rate, cashflows):
    return sum(cf / (1.0 + rate) ** i for i, cf in enumerate(cashflows))


def _irr(cashflows, guess: float = 0.1):
    lo, hi = -0.9999, 10.0
    f_lo, f_hi = _npv(lo, cashflows), _npv(hi, cashflows)
    if f_lo * f_hi <= 0:                       # bracketed -> bisection
        for _ in range(200):
            mid = (lo + hi) / 2.0
            f_mid = _npv(mid, cashflows)
            if abs(f_mid) < 1e-10:
                return mid
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo, f_lo = mid, f_mid
        return (lo + hi) / 2.0
    r = guess                                  # not bracketed -> Newton
    for _ in range(100):
        base = _npv(r, cashflows)
        deriv = (_npv(r + 1e-6, cashflows) - base) / 1e-6
        if deriv == 0:
            break
        step = base / deriv
        r -= step
        if abs(step) < 1e-10:
            return r
    return r


def _fv(rate, nper, pmt, pv=0.0):
    if rate == 0:
        return -(pv + pmt * nper)
    return -(pv * (1 + rate) ** nper + pmt * ((1 + rate) ** nper - 1) / rate)


def _pv(rate, nper, pmt, fv=0.0):
    if rate == 0:
        return -(fv + pmt * nper)
    return -(fv + pmt * ((1 + rate) ** nper - 1) / rate) / (1 + rate) ** nper


def _pmt(rate, nper, pv, fv=0.0):
    if rate == 0:
        return -(pv + fv) / nper
    return -(pv * (1 + rate) ** nper + fv) * rate / ((1 + rate) ** nper - 1)


def calc(expression: str) -> dict:
    """Evaluate a Python/finance math expression and return the numeric result.

    Use this for ALL arithmetic instead of computing in your head. The namespace
    exposes every `math.*` function plus Excel-style finance helpers:

      npv(rate, cashflows)         net present value; cashflows[0] is t=0
      irr(cashflows)               internal rate of return
      fv(rate, nper, pmt, pv=0)    future value of an annuity
      pv(rate, nper, pmt, fv=0)    present value of an annuity
      pmt(rate, nper, pv, fv=0)    periodic payment for a loan/annuity

    Examples:
      calc("npv(0.10, [-1000, 300, 400, 500, 600])")
      calc("irr([-1000, 300, 400, 500, 600])")
      calc("pmt(0.05/12, 360, 400000)")

    Args:
      expression: a single Python expression to evaluate.

    Returns:
      dict with "status" ("ok"/"error") and "result" or "error".
    """
    log.info("[calc] %s", expression[:160])
    ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    ns.update({
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
        "len": len, "pow": pow, "range": range, "sorted": sorted,
        "npv": _npv, "irr": _irr, "fv": _fv, "pv": _pv, "pmt": _pmt,
    })
    try:
        result = eval(expression, {"__builtins__": {}}, ns)  # sandboxed
        return {"status": "ok", "result": str(result)}
    except Exception as e:
        return {"status": "error", "error": f"{e}  (expression: {expression!r})"}


# quick self-check (no agent / no GCP needed)
print(calc("npv(0.10, [-1000, 300, 400, 500, 600])"))
print(calc("pmt(0.05/12, 360, 400000)"))

## System instruction

Defines the capital-allocation analyst persona and method: orient → plan →
query carefully → compute investment metrics with `calc` → rank & recommend an
allocation → explain business meaning/caveats → cite tables.

In [ ]:
INSTRUCTION = """\
You are a senior analyst supporting a CAPITAL ALLOCATION process. You have
direct, READ-ONLY access to a BigQuery data warehouse and a precise
financial-math calculator. Your analysis informs how limited capital is
deployed across competing projects, business units, segments, or assets.

Work like an investment-committee analyst, not a chatbot:

1. ORIENT. When you don't yet know the data, discover it first. Use the
   BigQuery tools to list datasets, list tables, and inspect table schemas
   BEFORE writing any SQL. Never invent table or column names. Identify the
   tables holding the capital-allocation inputs you need: cash flows,
   revenue/cost, invested capital, asset/segment identifiers, dates, and any
   risk-weighting or cost-of-capital references.

2. PLAN. State the candidate units (projects/segments/assets) you will
   compare, the tables and grain of each, and how you will join them. Confirm
   join keys from the schema, not from assumption.

3. QUERY CAREFULLY. All access is read-only (SELECT/WITH only). Be cost-aware:
   select only needed columns, filter by partition/date, and aggregate in SQL
   rather than pulling raw rows. Validate assumptions with small probing
   queries before large aggregations.

4. COMPUTE EXACTLY with `calc`. Do not do arithmetic in your head. For each
   candidate derive the metrics the question calls for, e.g.:
     - NPV at the relevant discount / hurdle rate      -> calc("npv(...)")
     - IRR of the cash-flow stream                     -> calc("irr(...)")
     - payback period, ROIC / RAROC, profitability index
     - return spread vs. cost of capital (EVA-style)
   Be explicit about the discount / hurdle rate you use and where it came from
   (a warehouse reference table if available, otherwise a stated assumption).

5. RANK & RECOMMEND. Compare candidates on a consistent basis, rank them, and
   give a clear allocation recommendation. Respect any stated capital budget or
   constraints; if a budget is given, allocate to maximize value within it and
   note what is funded vs. deferred and why.

6. EXPLAIN BUSINESS MEANING & CAVEATS. Describe what each table represents and
   the business rules you observed (status filters, currency handling,
   soft-deletes, capitalized vs. expensed items, period cut-offs). Flag data
   quality, coverage, or comparability issues that could change the ranking.

7. CITE. Reference the exact `project.dataset.table` and columns behind every
   figure so the committee can reproduce it.

If a request is ambiguous (discount rate, horizon, or capital budget
unspecified), state the assumption you're making and proceed; surface anything
that would materially change the allocation decision.
"""
print(INSTRUCTION[:200], "...")

## Build the agent — Gemini 2.5 (Vertex AI) + BigQuery toolset + `calc`

ADK's first-party **BigQuery toolset** gives the model `list_dataset_ids`,
`get_dataset_info`, `list_table_ids`, `get_table_info`, and `execute_sql`.
We run it with `WriteMode.BLOCKED` so it is strictly read-only. Application
Default Credentials drive both Vertex AI and BigQuery.

In [ ]:
import google.auth
from google.adk.agents import LlmAgent
from google.adk.planners import BuiltInPlanner
from google.adk.tools.bigquery import BigQueryCredentialsConfig, BigQueryToolset
from google.adk.tools.bigquery.config import BigQueryToolConfig, WriteMode
from google.genai import types


def build_agent() -> LlmAgent:
    credentials, adc_project = google.auth.default()
    project = PROJECT or adc_project or ""
    if not project:
        raise RuntimeError(
            "No GCP project. Set GOOGLE_CLOUD_PROJECT or run "
            "`gcloud auth application-default login`."
        )
    os.environ.setdefault("GOOGLE_CLOUD_PROJECT", project)
    log.info("Vertex project=%s location=%s model=%s", project, LOCATION, MODEL)

    # Read-only BigQuery toolset (writes BLOCKED).
    bq_tool_config = BigQueryToolConfig(
        write_mode=WriteMode.BLOCKED,
        max_query_result_rows=BQ_MAX_RESULT_ROWS,
    )
    bq_credentials = BigQueryCredentialsConfig(credentials=credentials)
    bigquery_toolset = BigQueryToolset(
        credentials_config=bq_credentials,
        bigquery_tool_config=bq_tool_config,
    )

    # Gemini 2.5 thinking -- surfaces the model's planning/reasoning budget.
    planner = BuiltInPlanner(
        thinking_config=types.ThinkingConfig(
            include_thoughts=True,
            thinking_budget=THINKING_BUDGET,
        )
    )

    return LlmAgent(
        model=MODEL,
        name="capital_allocation_analyst",
        description="Ranks competing investments from a BigQuery warehouse and "
                    "recommends how to allocate a limited capital budget.",
        instruction=INSTRUCTION,
        planner=planner,
        tools=[bigquery_toolset, calc],
        generate_content_config=types.GenerateContentConfig(temperature=0.2),
    )


log.info("build_agent defined")

## Runner + `ask` helper

ADK drives the agent through a session + event pipeline. `ask()` streams tool
calls and thoughts to the log and returns the final text answer. Jupyter supports
top-level `await`, so we call these coroutines directly.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService


async def make_runner() -> tuple[Runner, str]:
    """Create an agent, an in-memory session, and a Runner. Returns
    (runner, session_id). Reuse the same session_id for follow-up questions
    so context carries across turns."""
    agent = build_agent()
    session_service = InMemorySessionService()
    session_id = f"s-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    return runner, session_id


async def ask(runner: Runner, session_id: str, question: str) -> str:
    """Send one question; log tool activity; return the final text answer."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_text = ""
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "function_call", None):
                    fc = part.function_call
                    log.info("  -> tool call: %s(%s)", fc.name, dict(fc.args or {}))
                elif getattr(part, "function_response", None):
                    log.info("  <- tool result: %s", part.function_response.name)
        if event.is_final_response() and event.content and event.content.parts:
            final_text = "".join(
                p.text for p in event.content.parts if getattr(p, "text", None)
            )
    return final_text


log.info("Runner + ask helper ready")

## Drive it

Set `GOOGLE_CLOUD_PROJECT` (and authenticate) first, then run. The agent will
orient itself by listing datasets/tables and inspecting schemas before it writes
any SQL. Reuse `runner, session_id` across cells for multi-turn follow-ups.

In [ ]:
# runner, session_id = await make_runner()
#
# answer = await ask(runner, session_id,
#     "List the available datasets and tables, then identify the tables holding "
#     "project/segment cash flows and invested capital. Tell me their grain and "
#     "the keys that join them.")
# print(answer)

In [ ]:
# Follow-up on the SAME session (context carries over):
#
# answer = await ask(runner, session_id,
#     "For each candidate project, pull the annual cash flows and compute NPV at "
#     "a 10% hurdle rate, IRR, and payback using calc. Rank them and recommend "
#     "how to allocate a 500M capital budget, noting what is funded vs. deferred.")
# print(answer)